In [ ]:
"""
================================================================================
GCM ENSEMBLE TEMPERATURE + PRECIPITATION ANOMALIES
   Coropuna (Tropical) | Osorno (Temperate) | Guanaco (Arid)

  - 9-GCM CMIP6 ensemble: BCC-CSM2-MR, CAMS-CSM1-0, CESM2, EC-Earth3-Veg,
    FGOALS-f3-L, GFDL-ESM4, MPI-ESM1-2-HR, MRI-ESM2-0, NorESM2-MM
  - 4 SSPs: 1-2.6 / 2-4.5 / 3-7.0 / 5-8.5
  - Anomalies:
      Temperature:    T(t) - T_baseline       [degC]
      Precipitation: (P(t) - P_baseline)/P_baseline * 100  [%]
  - Time series: ensemble mean +/- 1 sd across the 9 GCMs, annual
  - 2071-2100 climatology annotated on each panel

This script re-initialises gdirs from the OGGM Hub's L5 preprocessed files, then reads the cached
gcm_data NetCDFs that Phase 1 wrote there. If Phase 1's working
directory has been deleted, this script
will re-initialise the gdirs but the gcm_data files will not be there
anymore, in that case, the GCM processing step needs to be re-run.

OUTPUT:
  - climate_anomaly_outputs/climate_anomalies_3sites.png
  - climate_anomaly_outputs/{Site}_climate_anomalies.csv

================================================================================
"""

import os
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

import oggm.cfg as cfg
from oggm import utils, workflow, DEFAULT_BASE_URL
from oggm.shop import gcm_climate

# ---------------------------------------------------------------------------
# OGGM initialisation
# ---------------------------------------------------------------------------
cfg.initialize(logging_level='WARNING')
cfg.PARAMS['continue_on_error']        = True
cfg.PARAMS['min_ice_thick_for_length'] = 1
cfg.PARAMS['store_model_geometry']     = True

WORKING_DIR_TMPL = 'OGGM_{name}_run'

# ---------------------------------------------------------------------------
# Site definitions
# ---------------------------------------------------------------------------
SITES = {
    'Coropuna': {
        'label':       'Coropuna icefield',
        'climate':     'tropical',
        'prcp_metric': 'percent',     
        'rgi_ids': [
            'RGI60-16.01473', 'RGI60-16.01474', 'RGI60-16.01475',
            'RGI60-16.01476', 'RGI60-16.01477', 'RGI60-16.01478',
            'RGI60-16.01479', 'RGI60-16.01480', 'RGI60-16.01481',
            'RGI60-16.01482', 'RGI60-16.01483', 'RGI60-16.01484',
            'RGI60-16.01485', 'RGI60-16.01486', 'RGI60-16.01487',
            'RGI60-16.01488', 'RGI60-16.01489', 'RGI60-16.01490',
            'RGI60-16.01491', 'RGI60-16.01492', 'RGI60-16.01493',
            'RGI60-16.01494', 'RGI60-16.01607', 'RGI60-16.01608',
            'RGI60-16.01609', 'RGI60-16.01613',
        ],
    },
    'Osorno': {
        'label':       'Osorno volcano icecap',
        'climate':     'temperate',
        'prcp_metric': 'percent',
        'rgi_ids': [
            'RGI60-17.12386', 'RGI60-17.12388', 'RGI60-17.12390',
            'RGI60-17.12392', 'RGI60-17.12394',
        ],
    },
    'Guanaco': {
        'label':       'Guanaco glacier',
        'climate':     'arid',
        'prcp_metric': 'absolute_mm',  
                                      
        'rgi_ids': [
            'RGI60-17.14868', 'RGI60-17.14869',
        ],
    },
}

GCMS = [
    'BCC-CSM2-MR', 'CAMS-CSM1-0', 'CESM2', 'EC-Earth3-Veg',
    'FGOALS-f3-L', 'GFDL-ESM4', 'MPI-ESM1-2-HR',
    'MRI-ESM2-0', 'NorESM2-MM',
]
SSPS = ['ssp126', 'ssp245', 'ssp370', 'ssp585']

SSP_COLOURS = {
    'ssp126': 'green',     'ssp245': 'blue',
    'ssp370': '#ff7f0e',   'ssp585': 'red',
}
SSP_LABELS = {
    'ssp126': 'SSP1-2.6', 'ssp245': 'SSP2-4.5',
    'ssp370': 'SSP3-7.0', 'ssp585': 'SSP5-8.5',
}

BP_SSP = ('https://cluster.klima.uni-bremen.de/~oggm/cmip6/GCM/'
          '{gcm}/{gcm}_{ssp}_r1i1p1f1_pr.nc')
BT_SSP = ('https://cluster.klima.uni-bremen.de/~oggm/cmip6/GCM/'
          '{gcm}/{gcm}_{ssp}_r1i1p1f1_tas.nc')

BASELINE = (1995, 2014)
PROJ_END = (2071, 2100)
PLOT_START = 2020

OUTDIR = os.path.join(os.getcwd(), 'climate_anomaly_outputs')
os.makedirs(OUTDIR, exist_ok=True)

# ---------------------------------------------------------------------------
# Ensures that GCM data exists in a gdir
# ---------------------------------------------------------------------------
def ensure_gcm_data(gdirs, gcm, ssp):
    """
    Check whether all gdirs have the gcm_data file for this (gcm, ssp).
    If not, re-process from Bremen. Returns True if data is available.
    """
    fid = f'_CMIP6_{gcm}_{ssp}'
    missing = [g for g in gdirs
               if not os.path.exists(g.get_filepath('gcm_data',
                                                    filesuffix=fid))]
    if not missing:
        return True

    print(f'    {len(missing)}/{len(gdirs)} gdirs missing {fid}, '
          f'reprocessing...')
    try:
        ft = utils.file_downloader(BT_SSP.format(gcm=gcm, ssp=ssp))
        fp = utils.file_downloader(BP_SSP.format(gcm=gcm, ssp=ssp))
        if ft is None or fp is None:
            print(f'    SKIP: GCM file not on Bremen server')
            return False
        workflow.execute_entity_task(
            gcm_climate.process_cmip_data, gdirs,
            filesuffix=fid,
            fpath_temp=ft,
            fpath_precip=fp,
        )
        return True
    except Exception as e:
        print(f'    SKIP: reprocessing failed: {e}')
        return False

def read_gcm_climate(gdir, gcm, ssp):
    
    fid = f'_CMIP6_{gcm}_{ssp}'
    path = gdir.get_filepath('gcm_data', filesuffix=fid)
    if not os.path.exists(path):
        return None, None
    with xr.open_dataset(path) as ds:
        temp_m = ds['temp'].values
        prcp_m = ds['prcp'].values
        time_vals = ds['time'].values

    try:
        years = pd.DatetimeIndex(time_vals).year.values
    except (TypeError, ValueError):
        # cftime objects expose .year directly
        years = np.array([int(t.year) for t in time_vals])

    df = pd.DataFrame({'year': years, 'temp': temp_m, 'prcp': prcp_m})
    temp_ann = df.groupby('year')['temp'].mean()      # monthly mean
    prcp_ann = df.groupby('year')['prcp'].sum()       # annual total mm
    return temp_ann, prcp_ann

# ---------------------------------------------------------------------------
# Initialise gdirs for each site
# ---------------------------------------------------------------------------
print('Initialising glacier directories for each site...')
print('(re-uses Phase 1 cache if working dirs still exist)\n')

site_gdirs = {}
for name, site in SITES.items():
    if not site['rgi_ids']:
        print(f'  SKIP {name}: no RGI IDs')
        continue

    cfg.PATHS['working_dir'] = utils.gettempdir(
        WORKING_DIR_TMPL.format(name=name), reset=False)
    print(f'  {name:<10}  working_dir = {cfg.PATHS["working_dir"]}')
    gdirs = workflow.init_glacier_directories(
        site['rgi_ids'],
        from_prepro_level=5,
        prepro_base_url=DEFAULT_BASE_URL,
    )
    site_gdirs[name] = gdirs
    total_area = sum(g.rgi_area_km2 for g in gdirs)
    print(f'             {len(gdirs)} glaciers,  total area '
          f'{total_area:.2f} km^2')

# ---------------------------------------------------------------------------
# Build per-site, per-GCM, per-SSP anomaly time series
# ---------------------------------------------------------------------------
print(f'\nReading bias-corrected GCM climate from gcm_data files')
print(f'Baseline: {BASELINE[0]}-{BASELINE[1]}')
print(f'Plot range: {PLOT_START}-{PROJ_END[1]}\n')

# results[site_name][var][ssp] -> DataFrame indexed by year, columns = GCMs
results = {}
for site_name, gdirs in site_gdirs.items():
    site = SITES[site_name]
    cfg.PATHS['working_dir'] = utils.gettempdir(
        WORKING_DIR_TMPL.format(name=site_name), reset=False)
    print(f'=== {site_name} ===')
    results[site_name] = {'tas': {ssp: pd.DataFrame() for ssp in SSPS},
                          'pr':  {ssp: pd.DataFrame() for ssp in SSPS}}

    # Pre-compute area weights for the site-mean
    areas = np.array([g.rgi_area_km2 for g in gdirs])
    weights = areas / areas.sum()

    for gcm in GCMS:
        per_ssp_baselines = {}
        for ssp in SSPS:
            if not ensure_gcm_data(gdirs, gcm, ssp):
                continue

            # Read each glacier's climate, take area-weighted mean
            t_each = []
            p_each = []
            for g in gdirs:
                t_ann, p_ann = read_gcm_climate(g, gcm, ssp)
                if t_ann is None:
                    t_each.append(None); p_each.append(None)
                    continue
                t_each.append(t_ann)
                p_each.append(p_ann)

            valid = [i for i, t in enumerate(t_each) if t is not None]
            if not valid:
                continue
            # Align all glaciers to a common year index
            t_df = pd.concat([t_each[i] for i in valid], axis=1)
            p_df = pd.concat([p_each[i] for i in valid], axis=1)
            w   = weights[valid] / weights[valid].sum()
            t_site = (t_df * w).sum(axis=1)
            p_site = (p_df * w).sum(axis=1)

            # Baseline
            t_base_slice = t_site.loc[
                (t_site.index >= BASELINE[0])
                & (t_site.index <= BASELINE[1])]
            p_base_slice = p_site.loc[
                (p_site.index >= BASELINE[0])
                & (p_site.index <= BASELINE[1])]
            if t_base_slice.empty or p_base_slice.empty:
                yr_min = int(t_site.index.min())
                yr_max = int(t_site.index.max())
                print(f'  {gcm} {ssp}: baseline {BASELINE[0]}-{BASELINE[1]}'
                      f' not in file (years available {yr_min}-{yr_max})')
                continue
            t_base = float(t_base_slice.mean())
            p_base = float(p_base_slice.mean())
            per_ssp_baselines[ssp] = (t_base, p_base)

            # Anomaly time series (temperature is always degC absolute;
            # precipitation can be either % or absolute mm/yr depending
            # on site - configured via the 'prcp_metric' field)
            t_anom = t_site - t_base
            if site['prcp_metric'] == 'absolute_mm':
                p_anom = p_site - p_base
            else:
                p_anom = (p_site - p_base) / p_base * 100.0

            results[site_name]['tas'][ssp][gcm] = t_anom
            results[site_name]['pr' ][ssp][gcm] = p_anom

        if per_ssp_baselines:
            t0, p0 = list(per_ssp_baselines.values())[0]
            print(f'  {gcm:<15}  baseline T = {t0:6.2f} degC   '
                  f'P = {p0:7.1f} mm yr-1')

# ---------------------------------------------------------------------------
# Save per-site CSVs
# ---------------------------------------------------------------------------
print('\nSaving CSVs...')
for site_name, dat in results.items():
    rows = []
    site = SITES[site_name]
    for var in ('tas', 'pr'):
        for ssp in SSPS:
            df = dat[var][ssp]
            if df.empty:
                continue
            mean = df.mean(axis=1)
            std  = df.std(axis=1)
            if var == 'tas':
                unit = 'degC'
            elif site['prcp_metric'] == 'absolute_mm':
                unit = 'mm/yr'
            else:
                unit = 'percent'
            for yr in mean.index:
                rows.append({
                    'site':     site_name,
                    'variable': var,
                    'unit':     unit,
                    'ssp':      ssp,
                    'year':     int(yr),
                    'mean':     float(mean.loc[yr]),
                    'std':      float(std.loc[yr]),
                    'n_gcm':    int(df.loc[yr].count()),
                })
    if rows:
        path = os.path.join(OUTDIR, f'{site_name}_climate_anomalies.csv')
        pd.DataFrame(rows).to_csv(path, index=False)
        print(f'  {path}')

# ---------------------------------------------------------------------------
# Combined figure: 2 rows (T, P) x 3 columns (Coropuna, Osorno, Guanaco)
# ---------------------------------------------------------------------------
ORDER = ['Coropuna', 'Osorno', 'Guanaco']
ORDER = [s for s in ORDER if s in results
         and any(not results[s]['tas'][ssp].empty for ssp in SSPS)]

if not ORDER:
    print('\nNo data to plot.')
else:
    fig, axes = plt.subplots(2, len(ORDER),
                             figsize=(5 * len(ORDER), 9), sharex=True)
    if len(ORDER) == 1:
        axes = axes.reshape(2, 1)

    for col, site_name in enumerate(ORDER):
        site = SITES[site_name]
        ax_t = axes[0, col]
        ax_p = axes[1, col]
        ax_t.set_title(f'{site["label"]}\n({site["climate"]} Andes, '
                       f'N = {len(site["rgi_ids"])})',
                       fontsize=11, fontweight='bold')

        text_y_t = 0.02
        text_y_p = 0.02
        for ssp in SSPS:
            c = SSP_COLOURS[ssp]
            df_t = results[site_name]['tas'][ssp]
            df_p = results[site_name]['pr' ][ssp]
            if not df_t.empty:
                m = df_t.mean(axis=1)
                s = df_t.std(axis=1)
                m_plot = m.loc[(m.index >= PLOT_START)
                               & (m.index <= PROJ_END[1])]
                s_plot = s.loc[(s.index >= PLOT_START)
                               & (s.index <= PROJ_END[1])]
                ax_t.plot(m_plot.index, m_plot.values, color=c, lw=1.2,
                          label=SSP_LABELS[ssp])
                ax_t.fill_between(m_plot.index,
                                  m_plot - s_plot, m_plot + s_plot,
                                  color=c, alpha=0.15)
                end_m = m.loc[PROJ_END[0]:PROJ_END[1]].mean()
                end_s = s.loc[PROJ_END[0]:PROJ_END[1]].mean()
                ax_t.text(0.98, text_y_t,
                          f'{PROJ_END[0]}-{PROJ_END[1]}: '
                          f'{end_m:.1f} \u00b1 {end_s:.1f} \u00b0C',
                          color=c, fontsize=8, ha='right',
                          transform=ax_t.transAxes)
                text_y_t += 0.06
            if not df_p.empty:
                m = df_p.mean(axis=1)
                s = df_p.std(axis=1)
                m_plot = m.loc[(m.index >= PLOT_START)
                               & (m.index <= PROJ_END[1])]
                s_plot = s.loc[(s.index >= PLOT_START)
                               & (s.index <= PROJ_END[1])]
                ax_p.plot(m_plot.index, m_plot.values, color=c, lw=1.2,
                          label=SSP_LABELS[ssp])
                ax_p.fill_between(m_plot.index,
                                  m_plot - s_plot, m_plot + s_plot,
                                  color=c, alpha=0.15)
                end_m = m.loc[PROJ_END[0]:PROJ_END[1]].mean()
                end_s = s.loc[PROJ_END[0]:PROJ_END[1]].mean()
                if site['prcp_metric'] == 'absolute_mm':
                    unit_str = 'mm yr$^{-1}$'
                    end_label = (f'{PROJ_END[0]}-{PROJ_END[1]}: '
                                 f'{end_m:+.0f} \u00b1 {end_s:.0f} '
                                 f'{unit_str}')
                else:
                    end_label = (f'{PROJ_END[0]}-{PROJ_END[1]}: '
                                 f'{end_m:+.0f} \u00b1 {end_s:.0f}%')
                ax_p.text(0.98, text_y_p, end_label,
                          color=c, fontsize=8, ha='right',
                          transform=ax_p.transAxes)
                text_y_p += 0.06

        ax_t.axhline(0, color='grey', lw=0.5, ls='--')
        ax_p.axhline(0, color='grey', lw=0.5, ls='--')
        ax_t.set_xlim(PLOT_START, PROJ_END[1])
        ax_p.set_xlim(PLOT_START, PROJ_END[1])
        ax_t.grid(alpha=0.25)
        ax_p.grid(alpha=0.25)
        ax_p.set_xlabel('Year')

        # Per-column y-labels: temperature label only on leftmost column,
        # precipitation label on every column because the unit may differ
        # (% vs mm yr-1) between sites
        if col == 0:
            ax_t.set_ylabel('Temperature anomaly (\u00b0C)')
            ax_t.legend(fontsize=9, loc='upper left')
        if site['prcp_metric'] == 'absolute_mm':
            ax_p.set_ylabel('Precipitation anomaly (mm yr$^{-1}$)')
        else:
            ax_p.set_ylabel('Precipitation anomaly (%)')

    fig.suptitle(f'Climate projection anomalies (relative to '
                 f'{BASELINE[0]}-{BASELINE[1]}) - bias-corrected, '
                 f'lapse-rate-adjusted',
                 fontsize=12, fontweight='bold', y=1.00)
    fig.tight_layout()
    fig_path = os.path.join(OUTDIR, 'climate_anomalies_3sites.png')
    fig.savefig(fig_path, dpi=200, bbox_inches='tight')
    plt.show()
    print(f'\nFigure saved: {fig_path}')

print('\nDone.')
print(f'Outputs in: {OUTDIR}')